# 11장. LLM과 함께 분석 질문을 다듬기

이 노트북은 외부 LLM을 직접 호출하지 않습니다. Chapter 05에서 만든 processed 데이터를 사용해 **원본 행 없이 안전한 구조 Context, 검증 가능한 Prompt Template, 사람 검토 Checklist, 실제 사용 전에는 `not_executed`로 표시되는 Log Template**을 만듭니다.


## 학습 목표

- raw 데이터로 조용히 fallback하지 않고 processed 입력을 확인합니다.
- 실제 값 예시 없이 데이터 구조를 요약합니다.
- 직접 민감정보로 추정되는 컬럼명은 Safe Context에서 기본 제외합니다.
- 식별자 컬럼은 관계 설명을 위해 이름만 사용할 수 있지만 원본 값은 공유하지 않습니다.
- Prompt에 목적·구조·요청·제약·출력·검증 조건을 함께 작성합니다.
- Chapter 09/10과 같은 예측 시점·누수·선택/테스트 기준을 LLM 요청에도 적용합니다.
- 외부 문서의 지시문은 명령이 아니라 `untrusted data`로 취급합니다.
- 실제 LLM을 사용하지 않은 빈 Log를 사용 증거로 오해하지 않습니다.


## 1. 프로젝트 루트와 processed 입력 설정


In [ ]:
from pathlib import Path
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Chapter 11은 processed 데이터에서 시작합니다. 먼저 python scripts/preprocess_data.py를 실행하세요. 누락: '
        + ', '.join(str(path) for path in missing_files)
    )

print('프로젝트 루트:', PROJECT_ROOT)
print('processed 입력:', PROCESSED_DIR)
print('결과 폴더:', REPORT_DIR)


## 2. Safe Context와 Prompt 자료 생성

공통 모듈은 processed 4개 파일이 모두 없으면 중단합니다. raw 데이터로 자동 fallback하지 않습니다. 이 단계에서도 외부 LLM API는 호출하지 않습니다.


In [ ]:
from src.llm_prompt_analysis import run_llm_prompt_analysis

result = run_llm_prompt_analysis(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    report_dir=REPORT_DIR,
)

print('사용 데이터:', result['source_type'])
print('외부 LLM 호출: 없음')


## 3. 데이터셋·컬럼 구조와 민감정보 검토


In [ ]:
dataset_summary = result['dataset_summary']
column_summary = result['column_summary']
sensitive_review = result['sensitive_review']

display(dataset_summary)
display(column_summary)
display(sensitive_review)


`column_summary`에는 실제 값 예시가 없습니다. `share_raw_values`는 항상 `no`이며, 직접 민감정보로 추정되는 컬럼명은 `do_not_share_name_by_default`, ID 계열은 `share_name_only_no_values`로 표시합니다. `low_cardinality_review=True`는 값 자체가 노출됐다는 뜻이 아니라 소수 범주 집계 시 재식별 위험을 사람이 다시 보라는 신호입니다.


## 4. 외부 LLM용 Safe Context 확인


In [ ]:
safe_context_text = result['safe_context_text']
context_validation = result['context_validation']

print(safe_context_text)
display(context_validation)


Safe Context는 **자동 승인 자료가 아닙니다.** 다음을 사람이 다시 확인합니다.

- 조직에서 허용한 LLM 도구와 계정인가?
- 컬럼명 자체가 내부 업무나 민감 속성을 드러내지 않는가?
- 소수 집단·희귀 범주 집계가 개인을 추정하게 하지 않는가?
- 오류 메시지·파일 경로·내부 URL·Secret이 없는가?
- 외부 웹/PDF/이메일 안의 지시문을 실행 명령으로 따르지 않는가?


## 5. Prompt Template 검토


In [ ]:
prompt_templates = result['prompt_templates']
display(
    prompt_templates[[
        'step', 'purpose', 'prompt_version',
        'context_rule', 'human_review_required', 'validation_point'
    ]]
)


In [ ]:
for selected_step in ['회귀 코드 검토', '분류 코드 검토', '외부 문서 검토']:
    print('\n' + '=' * 70)
    print(selected_step)
    print('=' * 70)
    prompt = prompt_templates.loc[
        prompt_templates['step'] == selected_step, 'prompt'
    ].iloc[0]
    print(prompt)


회귀 Prompt는 Chapter 09의 **예측 시점 → target 재료 누수 제외 → 날짜 순서 분할 → train 내부 TimeSeriesSplit 선택 → frozen final test** 흐름을 따릅니다. 분류 Prompt는 Chapter 10의 **completed/cancelled 범위 → Feature Contract → validation model/threshold 선택 → frozen test** 흐름을 따릅니다.


## 6. LLM 답변 검증 Checklist


In [ ]:
checklist = result['checklist']
display(checklist)


검증은 네 층으로 봅니다: **입력·보안 → 코드/수치 → 모델 → 해석**. 체크박스가 비어 있는 상태는 검증 완료를 의미하지 않습니다.


## 7. LLM 사용 Log Template — 아직 실행하지 않음


In [ ]:
usage_log = result['usage_log'].copy()
display(usage_log)
assert usage_log['execution_status'].eq('not_executed').all()
assert usage_log['final_use'].eq('not_used').all()


`ch11_llm_usage_log.csv`는 파일명이 Log라고 되어 있어도 처음 생성될 때는 **빈 템플릿**입니다. 실제 LLM을 사용했다면 해당 행만 `execution_status=executed`로 바꾸고 `executed_at`, `provider`, `model`, `input_summary`, `response_summary`, `validation_result`, `revision_note`, `final_use`를 실제 기록으로 채웁니다.


In [ ]:
# 실제 사용 후 한 행을 수정하는 예시입니다. 아래 코드는 실행하지 않아도 됩니다.
# row = usage_log['step'].eq('분석 질문 생성')
# usage_log.loc[row, 'execution_status'] = 'executed'
# usage_log.loc[row, 'executed_at'] = '실제 실행 시각'
# usage_log.loc[row, 'provider'] = '실제 제공자'
# usage_log.loc[row, 'model'] = '실제 모델명'
# usage_log.loc[row, 'prompt_version'] = '2.0'
# usage_log.loc[row, 'input_summary'] = '개인정보 없는 safe context'
# usage_log.loc[row, 'response_summary'] = '실제 답변 요약'
# usage_log.loc[row, 'validation_result'] = '실제 검증 결과'
# usage_log.loc[row, 'revision_note'] = '사람이 수정한 내용'
# usage_log.loc[row, 'final_use'] = 'used / partial / not_used'


## 8. 생성 파일 확인


In [ ]:
for name, path in result['output_paths'].items():
    print(name, 'OK' if path.exists() else 'MISSING', path)


주요 Evidence에는 `ch11_safe_context_validation.csv`가 추가됩니다. Safe Context, Prompt Template, Checklist, Usage Log는 LLM의 답변 자체가 아니라 **안전한 입력 설계와 검증·기록을 위한 자료**입니다.


## 9. 전체 자료 다시 생성하기

프로젝트 루트에서 다음 명령으로 같은 템플릿과 검증 자료를 다시 생성할 수 있습니다.

```powershell
python scripts/run_llm_prompt_analysis.py
```

이 명령은 외부 LLM을 호출하지 않습니다.


## 정리

Chapter 11의 핵심은 좋은 문장을 만드는 프롬프트 기술이 아니라 **안전한 Context → 검증 조건이 있는 Prompt → 답변 검증 → 사람 수정 → 실제 사용 기록**입니다. 다음 장에서는 LLM이 만든 분석 코드를 실행 전후에 더 구체적으로 검증합니다.
